# Module 10 — Notebook 2: Per-Model Analysis

## Learning Objectives

By the end of this notebook, you will be able to:

1. Convert a list of model output dicts into a pandas DataFrame
2. Use `groupby` to aggregate statistics by model
3. Compute per-model counts, flag rates, and average response lengths

**Prerequisite:** This notebook uses pandas. If you haven't used it before, review Module 04 first.

## Why This Matters for AI Research Engineering

Overall statistics can hide important differences between models. In safety research, it's common to compare multiple model versions or configurations side by side: Which model flags more often? Which gives longer, more detailed answers? Which model is more likely to produce harmful content?

Grouped analysis — breaking stats down by model — is one of the most fundamental tools in model evaluation. It turns a single aggregate number into a comparison that can drive real decisions about which model to deploy.

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_keys, check_length

# Load the data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

# Convert to a DataFrame
df = pd.DataFrame(outputs)
print(df.shape)
print(df.dtypes)
df.head(3)

## Concept: GroupBy Aggregation for Research Analysis

Pandas `groupby` lets you split a DataFrame into groups by one or more columns, then apply an aggregation function to each group.

```python
# Pattern: df.groupby('column')['other_column'].agg_function()
df.groupby('model')['flagged'].mean()   # average flagged per model
df.groupby('model').size()              # count of rows per model
```

The result is a Series (or DataFrame) indexed by the group values. To get a plain Python dict, call `.to_dict()` at the end.

This pattern is extremely common in research pipelines: group by model, split, condition, or category, then compute a metric for each group.

## Worked Example: Flag Rate and Count per Model

In [ ]:
# Flag rate per model
flag_rates = df.groupby('model')['flagged'].mean().round(4)
print("Flag rate per model:")
print(flag_rates)

print()

# Count per model
counts = df.groupby('model').size()
print("Count per model:")
print(counts)

## Exercise 1: Per-Model Output Counts

Create a dict called `model_counts` that maps each model name to the number of outputs it produced.

**Hint:** Use `df.groupby('model').size().to_dict()`.

In [ ]:
# Exercise 1: Per-model output counts
# model_counts = ...

# YOUR CODE HERE


In [ ]:
# Check Exercise 1
check_type(model_counts, dict, "model_counts is a dict")
check_keys(model_counts, ['model-a-v1', 'model-b-v1'], "model_counts has the right keys")
check_equal(model_counts['model-a-v1'], 11, "model-a-v1 has 11 outputs")
check_equal(model_counts['model-b-v1'], 9, "model-b-v1 has 9 outputs")

## Exercise 2: Per-Model Flag Rates

Create a dict called `model_flag_rates` that maps each model name to its flag rate, rounded to 4 decimal places.

**Hint:** Use `df.groupby('model')['flagged'].mean().round(4).to_dict()`.

In [ ]:
# Exercise 2: Per-model flag rates
# model_flag_rates = ...

# YOUR CODE HERE


In [ ]:
# Check Exercise 2
check_approx(model_flag_rates['model-a-v1'], 0.0, 0.001, "model-a-v1 flag rate is ~0.0")
check_approx(model_flag_rates['model-b-v1'], 0.7778, 0.001, "model-b-v1 flag rate is ~0.7778")

## Exercise 3: Per-Model Average Response Length

First, add a new column to `df` called `response_len` containing the character length of each response. Then compute a dict `model_avg_len` mapping each model to its mean response length, rounded to 2 decimal places.

**Hint:**
```python
df['response_len'] = df['response'].str.len()
model_avg_len = df.groupby('model')['response_len'].mean().round(2).to_dict()
```

In [ ]:
# Exercise 3: Average response length per model
# df['response_len'] = ...
# model_avg_len = ...

# YOUR CODE HERE


In [ ]:
# Check Exercise 3
check_approx(model_avg_len['model-a-v1'], 94.18, 0.5, "model-a-v1 avg response len is ~94.18")
check_approx(model_avg_len['model-b-v1'], 71.78, 0.5, "model-b-v1 avg response len is ~71.78")

## Summary

In this notebook you:

- Converted a list of dicts to a pandas DataFrame
- Used `groupby` to compute per-model counts, flag rates, and average response lengths
- Discovered a striking difference: `model-b-v1` has a 77.8% flag rate vs. 0% for `model-a-v1`

**Key takeaway:** Aggregate statistics can mask huge differences between subgroups. Per-model analysis revealed that one model is almost entirely responsible for all flagged outputs — exactly the kind of finding that matters for safety evaluations.

**Next:** In Notebook 3, you'll build heuristic classifiers to detect patterns in the outputs automatically.